In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Text Miners (1)/Actual Work/'
print('Connected:', BASE)

import os
print(os.listdir(BASE))

Mounted at /content/drive
Connected: /content/drive/MyDrive/Text Miners (1)/Actual Work/


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Text Miners (1)/Actual Work/'

In [ ]:
import os
print(os.listdir(BASE))

['pre-processing', 'data', 'features', 'task1-sentiments', 'task2-topics', 'task3-combination', 'dashboard', 'README_.docx']


In [ ]:
import os
print(os.listdir(f'{BASE}/data'))

['raw_reviews.gsheet', 'EDA Portion', 'MergedCleaned.csv']


In [ ]:
!pip install nltk pyspellchecker ftfy --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...


Done


[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [ ]:
import pandas as pd

df = pd.read_csv(f'{BASE}/data/MergedCleaned.csv')
print('Shape:', df.shape)
print(df.head(3))

Shape: (380505, 4)
   listing_id neighbourhood        room_type  \
0       27934   Ratchathewi  Entire home/apt   
1       27934   Ratchathewi  Entire home/apt   
2       27934   Ratchathewi  Entire home/apt   

                                            comments  
0  We stayed in the apartment for a week and we e...  
1  My girlfriend and I recently stayed in Nuttee'...  
2  I stayed for one month at the condo and was re...  


In [ ]:
preprocess_code = '''
import re
import nltk
import ftfy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

# Stopwords — keep negations since they affect sentiment
STOPWORDS = set(stopwords.words("english"))
KEEP_WORDS = {"not", "no", "never", "very", "but", "however"}
STOPWORDS = STOPWORDS - KEEP_WORDS

lemmatizer = WordNetLemmatizer()

def fix_encoding(text):
    """Fix mojibake encoding issues."""
    if not isinstance(text, str):
        return ""
    return ftfy.fix_text(text)

def lowercase(text):
    """Lowercase all text."""
    return text.lower()

def remove_html(text):
    """Remove HTML tags like <br/>."""
    return re.sub(r"<[^>]+>", " ", text)

def remove_punctuation(text):
    """Remove punctuation and special characters."""
    return re.sub(r"[^a-zA-Z0-9\\s]", " ", text)

def tokenize(text):
    """Split text into tokens."""
    return word_tokenize(text)

def filter_ascii(tokens):
    """Remove non-ASCII tokens (garbled characters)."""
    return [t for t in tokens if t.isascii()]

def remove_stopwords(tokens):
    """Remove stopwords but keep negations."""
    return [t for t in tokens if t not in STOPWORDS]

def lemmatize(tokens):
    """Reduce words to base form e.g. running -> run."""
    return [lemmatizer.lemmatize(t) for t in tokens]

def remove_short_tokens(tokens, min_len=2):
    """Remove single character tokens."""
    return [t for t in tokens if len(t) >= min_len]

def clean_text(text):
    """
    Full preprocessing pipeline.

    Steps:
        1. Fix encoding (ftfy)
        2. Lowercase
        3. Remove HTML tags
        4. Remove punctuation
        5. Tokenize
        6. Filter non-ASCII tokens
        7. Remove stopwords (keeping negations)
        8. Lemmatize
        9. Remove short tokens

    Args:
        text (str): raw review text

    Returns:
        tuple: (cleaned_text as string, tokens as list)
    """
    text = fix_encoding(text)
    text = lowercase(text)
    text = remove_html(text)
    text = remove_punctuation(text)
    tokens = tokenize(text)
    tokens = filter_ascii(tokens)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    tokens = remove_short_tokens(tokens)
    cleaned_text = " ".join(tokens)
    return cleaned_text, tokens


def preprocess_dataframe(df, text_col="comments"):
    """
    Apply clean_text to an entire dataframe column.

    Adds two new columns:
        cleaned_text — preprocessed text as a string
        tokens       — preprocessed text as a list of tokens

    Args:
        df (pd.DataFrame): dataframe with a text column
        text_col (str): name of the column containing raw text

    Returns:
        pd.DataFrame: original dataframe with two new columns added
    """
    results = df[text_col].apply(clean_text)
    df["cleaned_text"] = results.apply(lambda x: x[0])
    df["tokens"]       = results.apply(lambda x: x[1])
    return df
'''

# Save to shared Drive
output_path = f'{BASE}/pre-processing/preprocess.py'
with open(output_path, 'w') as f:
    f.write(preprocess_code)

print(f'✅ preprocess.py saved to {output_path}')

✅ preprocess.py saved to /content/drive/MyDrive/Text Miners (1)/Actual Work//pre-processing/preprocess.py


In [ ]:
import sys
sys.path.append(f'{BASE}/pre-processing')
import preprocess

# Test on 5 sample reviews
test_df = df.head(5).copy()
test_df = preprocess.preprocess_dataframe(test_df)

print('Sample output:')
for _, row in test_df.iterrows():
    print(f'\n  ORIGINAL: {row["comments"][:100]}')
    print(f'  CLEANED:  {row["cleaned_text"]}')
    print(f'  TOKENS:   {row["tokens"][:10]}')

Sample output:

  ORIGINAL: We stayed in the apartment for a week and we enjoyed it very much. Nuttee is a very nice host, and s
  CLEANED:  stayed apartment week enjoyed very much nuttee very nice host best accommodate everything perfect apartment love view balcony apartment very modern spacious location very central 10 min walk bts station supermarket 10 min bus taxi central world shopping mall also lot food stall massage nearby definitely stay next visit bangkok highly recommended
  TOKENS:   ['stayed', 'apartment', 'week', 'enjoyed', 'very', 'much', 'nuttee', 'very', 'nice', 'host']

  ORIGINAL: My girlfriend and I recently stayed in Nuttee's condo for a month.  It is a beautiful condo, with a 
  CLEANED:  girlfriend recently stayed nuttee condo month beautiful condo great view great location near victory monument abundance wonderful street food right around corner well shopping perfect area base exploration bangkok nuttee town but still very accessible via email assistant suchart 